<a href="https://colab.research.google.com/github/sayandxzzz/sayandxzzz/blob/main/SentimentAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
df = pd.read_csv('/content/IMDB Dataset.csv')

df.head()

In [ ]:
print(df.shape)

df['sentiment'].value_counts()

In [ ]:
df.isnull().sum()

In [ ]:
lemmatizer = WordNetLemmatizer()

def clean_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z]', ' ', text)

    words = text.split()

    words = [lemmatizer.lemmatize(word)
             for word in words
             if word not in stopwords.words('english')]

    return ' '.join(words)

df['clean_review'] = df['review'].apply(clean_text)

In [ ]:
df['sentiment'] = df['sentiment'].map({
    'positive':1,
    'negative':0
})

In [ ]:
tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df['clean_review'])

y = df['sentiment']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
model = LogisticRegression()

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
print(confusion_matrix(y_test, y_pred))

In [ ]:
def predict_sentiment(text):

    cleaned = clean_text(text)

    vector = tfidf.transform([cleaned])

    prediction = model.predict(vector)[0]

    if prediction == 1:
        return "Positive 😊"

    else:
        return "Negative 😞"

In [ ]:
review = input("Enter Review: ")

print(predict_sentiment(review))

In [ ]:
import matplotlib.pyplot as plt

df['sentiment'].value_counts().plot(kind='bar')

plt.title("Sentiment Distribution")

plt.show()

In [ ]:
from wordcloud import WordCloud

positive_text = ' '.join(
    df[df['sentiment']==1]['clean_review']
)

wc = WordCloud(
    width=800,
    height=400
).generate(positive_text)

plt.imshow(wc)

plt.axis('off')

plt.show()